# NeuroXVocal — Full Training Pipeline on Google Colab

**Dataset used:** ADReSSo — only the `diagnosis` subset (AD vs CN binary classification).
- Training: `diagnosis-train/diagnosis/train/audio/ad/` and `cn/`
- Testing:  `diagnosis-test/diagnosis/test-dist/audio/`
- Segmentation CSVs and the progression subset are NOT used.

**Before running:** Go to `Runtime → Change runtime type → T4 GPU`

## 0. Check GPU

In [ ]:
import torch
print('GPU available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
else:
    print('WARNING: No GPU found. Go to Runtime -> Change runtime type -> T4 GPU')

## 1. Install Dependencies

In [ ]:
%%capture
!pip install openai-whisper librosa praat-parselmouth soundfile torchaudio
!pip install transformers==4.45.2 sentencepiece
!pip install scikit-learn pandas numpy tqdm joblib

## 2. Mount Google Drive & Clone Repo

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

if not os.path.exists('/content/NeuroXVocal'):
    !git clone https://github.com/homiyano/NeuroXVocal.git /content/NeuroXVocal
else:
    print('Repo already cloned')

%cd /content/NeuroXVocal
!ls

## 3. Dataset Paths

Upload your ADReSSo dataset to Google Drive first, then set the root path below.

Expected Drive layout (just upload the zip/folder as-is from ADReSSo):
```
MyDrive/ADReSSo/
    diagnosis-train/diagnosis/train/audio/
        ad/   <- adrso024.wav, adrso025.wav ...
        cn/   <- adrso002.wav, adrso003.wav ...
    diagnosis-test/diagnosis/test-dist/audio/
        adrsdt1.wav, adrsdt2.wav ...
```

In [ ]:
# ── SET THIS TO WHERE YOU UPLOADED ADReSSo ──────────────────────────
ADRESSO_ROOT = '/content/drive/MyDrive/ADReSSo'
# ────────────────────────────────────────────────────────────────────

AD_WAV_DIR   = os.path.join(ADRESSO_ROOT, 'diagnosis-train/diagnosis/train/audio/ad')
CN_WAV_DIR   = os.path.join(ADRESSO_ROOT, 'diagnosis-train/diagnosis/train/audio/cn')
TEST_WAV_DIR = os.path.join(ADRESSO_ROOT, 'diagnosis-test/diagnosis/test-dist/audio')

# Verify paths exist
for label, path in [('AD train', AD_WAV_DIR), ('CN train', CN_WAV_DIR), ('Test', TEST_WAV_DIR)]:
    exists = os.path.isdir(path)
    wavs   = len([f for f in os.listdir(path) if f.endswith('.wav')]) if exists else 0
    print(f'{label}: {"OK" if exists else "NOT FOUND"}  ({wavs} .wav files)  {path}')

# Local working dir (Colab disk, faster than Drive for I/O)
WORK_DIR = '/content/processed_data'
AD_WORK  = os.path.join(WORK_DIR, 'ad')
CN_WORK  = os.path.join(WORK_DIR, 'cn')
os.makedirs(AD_WORK, exist_ok=True)
os.makedirs(CN_WORK, exist_ok=True)

## 4. Step I — Data Extraction

### 4a. Transcribe Audio with Whisper

In [ ]:
import whisper, time
from pathlib import Path

def transcribe_dir(wav_dir, out_dir, model):
    Path(out_dir).mkdir(parents=True, exist_ok=True)
    wavs = sorted(Path(wav_dir).glob('*.wav'))
    print(f'  Found {len(wavs)} files in {wav_dir}')
    times = []
    for wav in wavs:
        txt_file = Path(out_dir) / (wav.stem + '.txt')
        if txt_file.exists():
            continue
        t0 = time.perf_counter()
        result = model.transcribe(str(wav))
        elapsed = time.perf_counter() - t0
        times.append(elapsed)
        txt_file.write_text(result['text'], encoding='utf-8')
        print(f'  {wav.name:<30}  {elapsed:.2f}s')
    if times:
        total = sum(times)
        print(f'  ── Total: {total:.1f}s  |  Avg per file: {total/len(times):.2f}s  |  Files: {len(times)}')
    print(f'  Done -> {out_dir}')

whisper_model = whisper.load_model('base')

print('Transcribing AD...')
t_start = time.perf_counter()
transcribe_dir(AD_WAV_DIR, AD_WORK, whisper_model)
print(f'AD transcription total: {time.perf_counter()-t_start:.1f}s\n')

print('Transcribing CN...')
t_start = time.perf_counter()
transcribe_dir(CN_WAV_DIR, CN_WORK, whisper_model)
print(f'CN transcription total: {time.perf_counter()-t_start:.1f}s')

### 4b. Extract Audio Features (librosa + parselmouth)

In [ ]:
import time

AD_FEATURES_RAW = os.path.join(AD_WORK, 'audio_features_ad.csv')
CN_FEATURES_RAW = os.path.join(CN_WORK, 'audio_features_cn.csv')

print('Extracting AD audio features...')
t0 = time.perf_counter()
!python /content/NeuroXVocal/src/data_extraction/extract_audio_features.py \
    {AD_WAV_DIR} --output_csv {AD_FEATURES_RAW}
print(f'AD features time: {time.perf_counter()-t0:.1f}s\n')

print('Extracting CN audio features...')
t0 = time.perf_counter()
!python /content/NeuroXVocal/src/data_extraction/extract_audio_features.py \
    {CN_WAV_DIR} --output_csv {CN_FEATURES_RAW}
print(f'CN features time: {time.perf_counter()-t0:.1f}s')

### 4c. Extract Audio Embeddings (Wav2Vec2)

In [ ]:
import time

AD_EMB_RAW = os.path.join(AD_WORK, 'audio_embeddings_ad.csv')
CN_EMB_RAW = os.path.join(CN_WORK, 'audio_embeddings_cn.csv')

print('Extracting AD audio embeddings (Wav2Vec2)...')
t0 = time.perf_counter()
!python /content/NeuroXVocal/src/data_extraction/extract_audio_embeddings.py \
    {AD_WAV_DIR} --output_csv {AD_EMB_RAW}
print(f'AD embeddings time: {time.perf_counter()-t0:.1f}s\n')

print('Extracting CN audio embeddings (Wav2Vec2)...')
t0 = time.perf_counter()
!python /content/NeuroXVocal/src/data_extraction/extract_audio_embeddings.py \
    {CN_WAV_DIR} --output_csv {CN_EMB_RAW}
print(f'CN embeddings time: {time.perf_counter()-t0:.1f}s')

## 5. Step II — Preprocessing

### 5a. Preprocess Transcriptions

In [ ]:
AD_TEXT_PROC = os.path.join(WORK_DIR, 'ad_text_processed')
CN_TEXT_PROC = os.path.join(WORK_DIR, 'cn_text_processed')

!python /content/NeuroXVocal/src/data_processing/preprocess_texts.py \
    {AD_WORK} {AD_TEXT_PROC}

!python /content/NeuroXVocal/src/data_processing/preprocess_texts.py \
    {CN_WORK} {CN_TEXT_PROC}

### 5b. Preprocess Audio Features

In [ ]:
import shutil, os

SCALER_FEATURES = '/content/NeuroXVocal/src/inference/scaler_params_audio_features.pkl'

AD_FEAT_PROC_DIR = os.path.join(WORK_DIR, 'ad_features_processed')
CN_FEAT_PROC_DIR = os.path.join(WORK_DIR, 'cn_features_processed')
os.makedirs(AD_FEAT_PROC_DIR, exist_ok=True)
os.makedirs(CN_FEAT_PROC_DIR, exist_ok=True)

!python /content/NeuroXVocal/src/data_processing/preprocess_audio_features.py \
    --input_path {AD_FEATURES_RAW} \
    --output_path {AD_FEAT_PROC_DIR} \
    --scaler_path {SCALER_FEATURES}

!python /content/NeuroXVocal/src/data_processing/preprocess_audio_features.py \
    --input_path {CN_FEATURES_RAW} \
    --output_path {CN_FEAT_PROC_DIR} \
    --scaler_path {SCALER_FEATURES}

AD_FEAT_FINAL = os.path.join(AD_WORK, 'audio_features_ad_processed.csv')
CN_FEAT_FINAL = os.path.join(CN_WORK, 'audio_features_cn_processed.csv')
shutil.copy(os.path.join(AD_FEAT_PROC_DIR, 'audio_features_ad.csv'), AD_FEAT_FINAL)
shutil.copy(os.path.join(CN_FEAT_PROC_DIR, 'audio_features_cn.csv'), CN_FEAT_FINAL)
print('Audio features preprocessed.')

### 5c. Preprocess Audio Embeddings

In [ ]:
import pandas as pd, numpy as np, joblib

SCALER_EMB = '/content/NeuroXVocal/src/inference/scaler_params_audio_emb.pkl'

def preprocess_embeddings(input_csv, output_csv, scaler_path):
    df = pd.read_csv(input_csv)
    patient_ids = df['patient_id'].values
    features = df.drop(columns=['patient_id'])
    features = features.apply(lambda x: x.fillna(x.mean()) if x.isnull().any() else x)
    scaler = joblib.load(scaler_path)
    scaled = scaler.transform(features)
    df_out = pd.DataFrame(scaled, columns=features.columns)
    df_out.insert(0, 'patient_id', patient_ids)
    df_out.to_csv(output_csv, index=False)
    print(f'Saved {output_csv}')

AD_EMB_FINAL = os.path.join(AD_WORK, 'audio_embeddings_ad_processed.csv')
CN_EMB_FINAL = os.path.join(CN_WORK, 'audio_embeddings_cn_processed.csv')

preprocess_embeddings(AD_EMB_RAW, AD_EMB_FINAL, SCALER_EMB)
preprocess_embeddings(CN_EMB_RAW, CN_EMB_FINAL, SCALER_EMB)

## 6. Verify Everything is Ready

In [ ]:
from pathlib import Path

checks = [
    ('AD features',   AD_FEAT_FINAL),
    ('CN features',   CN_FEAT_FINAL),
    ('AD embeddings', AD_EMB_FINAL),
    ('CN embeddings', CN_EMB_FINAL),
]
for name, path in checks:
    df = pd.read_csv(path)
    print(f'{name}: {df.shape[0]} patients, {df.shape[1]-1} features')

# Check all feature patients have matching transcripts
ad_feat_ids = set(pd.read_csv(AD_FEAT_FINAL)['patient_id'].astype(str))
cn_feat_ids = set(pd.read_csv(CN_FEAT_FINAL)['patient_id'].astype(str))
ad_txt_ids  = {p.stem for p in Path(AD_TEXT_PROC).glob('*.txt')}
cn_txt_ids  = {p.stem for p in Path(CN_TEXT_PROC).glob('*.txt')}

ad_miss = ad_feat_ids - ad_txt_ids
cn_miss = cn_feat_ids - cn_txt_ids
if ad_miss: print(f'WARNING: {len(ad_miss)} AD patients missing transcripts:', ad_miss)
if cn_miss: print(f'WARNING: {len(cn_miss)} CN patients missing transcripts:', cn_miss)
if not ad_miss and not cn_miss:
    print('All patients have transcripts. Ready to train!')

## 7. Configure & Train

In [ ]:
RESULTS_DIR = '/content/NeuroXVocal/results'
os.makedirs(RESULTS_DIR, exist_ok=True)

config = f'''import os

AD_TEXT_DIR        = r"{AD_TEXT_PROC}"
CN_TEXT_DIR        = r"{CN_TEXT_PROC}"
AD_CSV             = r"{AD_FEAT_FINAL}"
CN_CSV             = r"{CN_FEAT_FINAL}"
AD_EMBEDDING_CSV   = r"{AD_EMB_FINAL}"
CN_EMBEDDING_CSV   = r"{CN_EMB_FINAL}"

TEXT_EMBEDDING_MODEL   = 'microsoft/deberta-v3-base'
NUM_MFCC_FEATURES      = 47
NUM_EMBEDDING_FEATURES = 768
AUDIO_CHANNELS         = 1
CUDA                   = True

BATCH_SIZE              = 8
EPOCHS                  = 200
LEARNING_RATE           = 1e-5
WEIGHT_DECAY            = 1e-4
NUM_FOLDS               = 5
SAVE_BEST_MODEL         = True
EARLY_STOPPING_PATIENCE = 10

SAVE_MODEL_PATH = r"{RESULTS_DIR}/model"
LOG_PATH        = r"{RESULTS_DIR}/training.log"
'''

with open('/content/NeuroXVocal/src/train/config.py', 'w') as f:
    f.write(config)
print('Config written. Starting training...')

In [ ]:
# ~2-4 hours on T4. Each fold saves its best checkpoint to results/
%cd /content/NeuroXVocal/src/train
!python main.py

## 8. Training & Validation Loss Plot

In [ ]:
import re
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

LOG_PATH = f'{RESULTS_DIR}/training.log'

# ── Parse training.log ───────────────────────────────────────────────
# Each line: "Fold X, Epoch Y, Train Loss: A, Val Loss: B"
pattern = re.compile(
    r'Fold (\d+), Epoch (\d+), Train Loss: ([\d.]+), Val Loss: ([\d.]+)'
)

folds = {}   # fold_num -> {'train': [], 'val': [], 'epochs': []}
with open(LOG_PATH) as f:
    for line in f:
        m = pattern.search(line)
        if m:
            fold, epoch, train_loss, val_loss = int(m.group(1)), int(m.group(2)), float(m.group(3)), float(m.group(4))
            if fold not in folds:
                folds[fold] = {'epochs': [], 'train': [], 'val': []}
            folds[fold]['epochs'].append(epoch)
            folds[fold]['train'].append(train_loss)
            folds[fold]['val'].append(val_loss)

num_folds = len(folds)
print(f'Parsed {num_folds} folds from log.')
for fold, data in sorted(folds.items()):
    best_epoch = data['epochs'][data['val'].index(min(data['val']))]
    print(f'  Fold {fold}: {len(data["epochs"])} epochs  |  best val loss = {min(data["val"]):.4f} at epoch {best_epoch}')

# ── Plot ─────────────────────────────────────────────────────────────
cols = min(num_folds, 3)
rows = (num_folds + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(6 * cols, 4 * rows))
fig.suptitle('NeuroXVocal — Training vs Validation Loss per Fold', fontsize=14, fontweight='bold', y=1.02)

# flatten axes array for easy indexing
axes_flat = axes.flatten() if num_folds > 1 else [axes]

for idx, (fold, data) in enumerate(sorted(folds.items())):
    ax = axes_flat[idx]
    epochs     = data['epochs']
    train_loss = data['train']
    val_loss   = data['val']

    best_idx   = val_loss.index(min(val_loss))
    best_epoch = epochs[best_idx]
    best_val   = val_loss[best_idx]

    ax.plot(epochs, train_loss, color='#2196F3', linewidth=2, label='Train loss')
    ax.plot(epochs, val_loss,   color='#F44336', linewidth=2, label='Val loss')

    # mark best epoch
    ax.axvline(x=best_epoch, color='#4CAF50', linestyle='--', linewidth=1.5, alpha=0.8)
    ax.scatter([best_epoch], [best_val], color='#4CAF50', s=80, zorder=5)
    ax.annotate(f'best\nepoch {best_epoch}\nloss {best_val:.4f}',
                xy=(best_epoch, best_val),
                xytext=(best_epoch + 0.5, best_val + (max(val_loss) - min(val_loss)) * 0.15),
                fontsize=8, color='#2E7D32',
                arrowprops=dict(arrowstyle='->', color='#2E7D32', lw=1.2))

    ax.set_title(f'Fold {fold}', fontweight='bold')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)
    ax.set_xlim(left=1)

# hide unused subplots if any
for idx in range(num_folds, len(axes_flat)):
    axes_flat[idx].set_visible(False)

plt.tight_layout()

plot_path = f'{RESULTS_DIR}/loss_curves.png'
plt.savefig(plot_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Plot saved to {plot_path}')

## 9. All-Folds Overlay (Train & Val Loss on One Chart)

In [ ]:
FOLD_COLORS = ['#E53935', '#8E24AA', '#1E88E5', '#00897B', '#F4511E']

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('All Folds Overlay', fontsize=13, fontweight='bold')

for idx, (fold, data) in enumerate(sorted(folds.items())):
    color  = FOLD_COLORS[idx % len(FOLD_COLORS)]
    epochs = data['epochs']
    ax1.plot(epochs, data['train'], color=color, linewidth=1.8, label=f'Fold {fold}')
    ax2.plot(epochs, data['val'],   color=color, linewidth=1.8, label=f'Fold {fold}')

    # mark best val epoch on val plot
    best_idx = data['val'].index(min(data['val']))
    ax2.scatter([epochs[best_idx]], [data['val'][best_idx]], color=color, s=70, zorder=5)

for ax, title in [(ax1, 'Training Loss'), (ax2, 'Validation Loss')]:
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)
    ax.set_xlim(left=1)

plt.tight_layout()
overlay_path = f'{RESULTS_DIR}/loss_overlay.png'
plt.savefig(overlay_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Overlay plot saved to {overlay_path}')

## 10. Save Results to Drive (in case Colab disconnects)

In [ ]:
import glob, shutil

model_files = sorted(glob.glob(f'{RESULTS_DIR}/*.pth'))
print('Saved model checkpoints:')
for m in model_files:
    print(f'  {os.path.basename(m)}  ({os.path.getsize(m)/1e6:.1f} MB)')

DRIVE_RESULTS = '/content/drive/MyDrive/NeuroXVocal_results'
os.makedirs(DRIVE_RESULTS, exist_ok=True)
for f in glob.glob(f'{RESULTS_DIR}/*'):
    shutil.copy(f, os.path.join(DRIVE_RESULTS, os.path.basename(f)))
    print(f'  Copied {os.path.basename(f)} to Drive')

## 11. Download Best Model to Your Mac

In [ ]:
from google.colab import files
import zipfile

zip_path = '/content/neuroxvocal_models.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for f in glob.glob(f'{RESULTS_DIR}/*.pth'):
        zf.write(f, os.path.basename(f))
    for f in glob.glob(f'{RESULTS_DIR}/*.png'):
        zf.write(f, os.path.basename(f))
    log = f'{RESULTS_DIR}/training.log'
    if os.path.exists(log):
        zf.write(log, 'training.log')

print('Downloading zip...')
files.download(zip_path)

## 12. View Training Log

In [ ]:
log_path = f'{RESULTS_DIR}/training.log'
if os.path.exists(log_path):
    with open(log_path) as f:
        lines = f.readlines()
    print(''.join(lines[-80:]))
else:
    print('Log not found yet.')

---
## After Training — Use Locally

1. Unzip the downloaded file — you'll get files like `model_fold1_best.pth` ... `model_fold5_best.pth`
2. Pick the best fold (check `training.log` for lowest val loss)
3. On your Mac:
```bash
mkdir -p "/Users/homi/Documents/Vibe Coding/NeuroXVocal/results"
cp model_fold2_best.pth "/Users/homi/Documents/Vibe Coding/NeuroXVocal/results/best.pth"
```
4. Run the app:
```bash
cd "/Users/homi/Documents/Vibe Coding/NeuroXVocal/app"
../venv/bin/python -m streamlit run neuroxvocal_app.py
```